In [ ]:

import base64, json, os, pathlib, signal, subprocess, sys, time, torch
T0 = time.monotonic(); CAP = 14400.0; START_CUTOFF = 11400.0
w = pathlib.Path("/kaggle/working"); p = w / "payload"; p.mkdir(exist_ok=True)
BLOB = {'pikan_campaign.py': '...', 'dfg_long.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiRFJBRlRfVU5WRVJJRklFRCAoTE9ORyBSRUNJUEUgdmFyaWFudCBvZiBkZmdfZGFuaWVsLnB5LCAyMDI2LTEwLTAxKTogc2FtZSBwcm9ibGVtL2FyY2hpdGVjdHVyZXMvcG9vbHMgYXMgdGhlIERhbmllbC1jb25maWcgcnVuLCBidXQgQWRhbSAyMDAwMCBzdGVwcw0KdGhlbiByb2J1c3Qgc3Ryb25nLVdvbGZlIEwtQkZHUyAobWF4X2l0ZXIgMjUsIG1heF9ldmFsIDEyNSwgaGlzdG9yeSAxMDApIHdpdGggdGltZSBjYXAsIHBsYXRlYXUgYW5kIHplcm8tcHJvZ3Jlc3Mgc3RvcHMuDQpPUklHSU5BTCBET0M6IERGRyAyRC0xIChSZT0yMCwgc3RlYWR5KSB1bmRlciB0aGUgY29uZmlndXJhdGlvbiBvZiBHb256YWxleiBDYXJyaWxsbydzIDIwMjQgVVZHIHRoZXNpcyAoc2VjdGlvbiA1LjEuMik6DQpjaGFubmVsIDIuMiB4IDAuNDEsIGN5bGluZGVyIHI9MC4wNSBhdCAoMC4yLDAuMiksIG51PTAuMDAxLCBpbmxldCB1PTQqMC4zKnkqKDAuNDEteSkvMC40MV4yLCBMSFMgc2FtcGxpbmcgKDMxOCBib3VuZGFyeSBwb2ludHMsDQoyMiwzNDUgY29sbG9jYXRpb24gcG9pbnRzKSwgTUxQIDR4NTAgdGFuaCB3aXRoIDMgcmF3IG91dHB1dHMgKHUsdixwKSwgbG9zcyA9IE1TRShOU3gpK01TRShOU3kpK01TRShjb250KSArIE1TRShCQyksDQpBZGFtIDIwMDAgZXBvY2hzIGxyIDFlLTMgdGhlbiBMLUJGR1MgMTAwMCBpdGVyYXRpb25zIGxyIDAuMSAoaW50ZXJwcmV0ZWQgYXMgMTAwMCBzaW5nbGUtaXRlcmF0aW9uIHN0ZXBzLCBubyBsaW5lIHNlYXJjaCkuDQpWYXJpYW50czoNCiAgbGl0ZXJhbCAgIDogYm91bmRhcnkgcG9pbnRzIG9uIGlubGV0LCB3YWxscyBhbmQgY3lsaW5kZXIgT05MWSwgcHJlc3N1cmUgdGFyZ2V0IDAgYXQgQUxMIG9mIHRoZW0gKGFzIGRyYXduIGluIHRoZSB0aGVzaXMsIEZpZy4gMTkpOw0KICAgICAgICAgICAgICBubyBvdXRsZXQgY29uZGl0aW9uLiAgS2VwdCB0byBzaG93IHdoYXQgdGhlIHB1Ymxpc2hlZCBzZXR1cCB5aWVsZHM7IGl0IGlzIG5vdCB0aGUgREZHIGJlbmNobWFyay4NCiAgY29ycmVjdGVkIDogdmVsb2NpdHkgQkMgb24gaW5sZXQvd2FsbHMvY3lsaW5kZXIsIEZFQVRGTE9XIGRvLW5vdGhpbmcgb3V0bGV0IChudSp1X3ggLSBwID0gMCwgbnUqdl94ID0gMCkgb24gdGhlIG91dGxldCwgbm8gcCBwaW5zLg0KQXJtczogbWxwICg0eDUwLCByYXcgY29vcmRpbmF0ZXMpIGFuZCByYmYgKFJCRi1QSUtBTiwgd2lkdGggLS13aWR0aCwgYmFzZXMgMTEsIHNhbWUgcmF3IG91dHB1dHMpLiAgUG9vbHMgc2hhcmVkIHBlciBzZWVkLg0KU2NvcmVkIHBvc3QgaG9jIGFnYWluc3QgdGhlIEZFTSBERkctTDMgd29ya2luZyByZWZlcmVuY2UgKDIyMXg0MiBncmlkLCBvdXRsZXQtZ2F1Z2UgcHJlc3N1cmUpIGFuZCB3aXRoIHRoZSBGRUFURkxPVyBzaWdtYS5uIFFvSXMuDQpNb2Rlczogd29ya2VyIHwgY2FtcGFpZ24uDQoiIiINCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQgaGFzaGxpYg0KaW1wb3J0IGpzb24NCmltcG9ydCBtYXRoDQppbXBvcnQgc3VicHJvY2Vzcw0KaW1wb3J0IHN5cw0KaW1wb3J0IHRpbWUNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCB0b3JjaA0KZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgcW1jDQoNCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCkpDQpmcm9tIHBpa2FuX2NhbXBhaWduIGltcG9ydCBNTFAsIFJCRktBTiwgZ3JhZCAgIyBub3FhOiBFNDAyDQoNClNUQVRVUyA9ICJEUkFGVF9VTlZFUklGSUVEIg0KTCwgSCwgQ1gsIENZLCBSID0gMi4yLCAwLjQxLCAwLjIsIDAuMiwgMC4wNQ0KTlUsIFVNQVgsIFVNRUFOLCBESUFNID0gMC4wMDEsIDAuMywgMC4yLCAwLjENCkZFQVQgPSB7IkNkIjogNS41Nzk1MzUyMzM4NCwgIkNsIjogMC4wMTA2MTg5NDgxNDYsICJkcCI6IDAuMTE3NTIwMTY2OTd9DQpOX0JDLCBOX0NPTCA9IDMxOCwgMjIzNDUNCg0KDQpkZWYgbGhzKG4sIGQsIHNlZWQpOg0KICAgIHJldHVybiBxbWMuTGF0aW5IeXBlcmN1YmUoZD1kLCBzZWVkPXNlZWQpLnJhbmRvbShuKQ0KDQoNCmRlZiBtYWtlX3Bvb2xzKHNlZWQsIHZhcmlhbnQsIGR0eXBlLCBkZXZpY2UpOg0KICAgIHB0cyA9IGxocygzMjAwMCwgMiwgMTAgKiBzZWVkICsgMSkgKiBucC5hcnJheShbTCwgSF0pDQogICAgcHRzID0gcHRzW25wLmh5cG90KHB0c1s6LCAwXSAtIENYLCBwdHNbOiwgMV0gLSBDWSkgPiBSXVs6Tl9DT0xdDQogICAgYXNzZXJ0IGxlbihwdHMpID09IE5fQ09MDQogICAgbGVuZ3RocyA9IHsiaW5sZXQiOiBILCAidG9wIjogTCwgImJvdHRvbSI6IEwsICJjeWwiOiAyICogbWF0aC5waSAqIFJ9DQogICAgaWYgdmFyaWFudCA9PSAiY29ycmVjdGVkIjoNCiAgICAgICAgbGVuZ3Roc1sib3V0bGV0Il0gPSBIDQogICAgYmFzZSA9IHtrOiB2IGZvciBrLCB2IGluIGxlbmd0aHMuaXRlbXMoKSBpZiBrICE9ICJvdXRsZXQifQ0KICAgIHRvdCA9IHN1bShiYXNlLnZhbHVlcygpKQ0KICAgIGNvdW50cyA9IHtrOiBpbnQocm91bmQoTl9CQyAqIHYgLyB0b3QpKSBmb3IgaywgdiBpbiBiYXNlLml0ZW1zKCl9DQogICAgY291bnRzWyJ0b3AiXSArPSBOX0JDIC0gc3VtKGNvdW50cy52YWx1ZXMoKSkNCiAgICBpZiB2YXJpYW50ID09ICJjb3JyZWN0ZWQiOg0KICAgICAgICBjb3VudHNbIm91dGxldCJdID0gbWF4KDgsIGludChyb3VuZChOX0JDICogSCAvIHRvdCkpKQ0KICAgIGJjID0ge30NCiAgICBmb3IgaSwgKGssIG4pIGluIGVudW1lcmF0ZShjb3VudHMuaXRlbXMoKSk6DQogICAgICAgIHMgPSBsaHMobiwgMSwgMTAgKiBzZWVkICsgMyArIDEwMCAqIGkpWzosIDBdDQogICAgICAgIGlmIGsgPT0gImlubGV0IjogeHkgPSBucC5zdGFjaygobnAuemVyb3MobiksIHMgKiBIKSwgMSkNCiAgICAgICAgZWxpZiBrID09ICJvdXRsZXQiOiB4eSA9IG5wLnN0YWNrKChucC5mdWxsKG4sIEwpLCBzICogSCksIDEpDQogICAgICAgIGVsaWYgayA9PSAidG9wIjogeHkgPSBucC5zdGFjaygocyAqIEwsIG5wLmZ1bGwobiwgSCkpLCAxKQ0KICAgICAgICBlbGlmIGsgPT0gImJvdHRvbSI6IHh5ID0gbnAuc3RhY2soKHMgKiBMLCBucC56ZXJvcyhuKSksIDEpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICB0aCA9IDIgKiBtYXRoLnBpICogcw0KICAgICAgICAgICAgeHkgPSBucC5zdGFjaygoQ1ggKyBSICogbnAuY29zKHRoKSwgQ1kgKyBSICogbnAuc2luKHRoKSksIDEpDQogICAgICAgIGJjW2tdID0geHkNCiAgICB0ID0gbGFtYmRhIGE6IHRvcmNoLmFzX3RlbnNvcihhLCBkdHlwZT1kdHlwZSwgZGV2aWNlPWRldmljZSkNCiAgICByZXR1cm4geyJjb2wiOiB0KHB0cyksICoqe2s6IHQodikgZm9yIGssIHYgaW4gYmMuaXRlbXMoKX19LCB7azogbGVuKHYpIGZvciBrLCB2IGluIGJjLml0ZW1zKCl9DQoNCg0KZGVmIHJlc2lkdWFscyhtb2RlbCwgeHkpOg0KICAgIHh5ID0geHkuZGV0YWNoKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkNCiAgICBvID0gbW9kZWwoeHkpDQogICAgdSwgdiwgcCA9IG9bOiwgMDoxXSwgb1s6LCAxOjJdLCBvWzosIDI6M10NCiAgICBndSwgZ3YsIGdwID0gZ3JhZCh1LCB4eSksIGdyYWQodiwgeHkpLCBncmFkKHAsIHh5KQ0KICAgIHV4eCA9IGdyYWQoZ3VbOiwgMDoxXSwgeHkpWzosIDA6MV07IHV5eSA9IGdyYWQoZ3VbOiwgMToyXSwgeHkpWzosIDE6Ml0NCiAgICB2eHggPSBncmFkKGd2WzosIDA6MV0sIHh5KVs6LCAwOjFdOyB2eXkgPSBncmFkKGd2WzosIDE6Ml0sIHh5KVs6LCAxOjJdDQogICAgbnN4ID0gdSAqIGd1WzosIDA6MV0gKyB2ICogZ3VbOiwgMToyXSArIGdwWzosIDA6MV0gLSBOVSAqICh1eHggKyB1eXkpDQogICAgbnN5ID0gdSAqIGd2WzosIDA6MV0gKyB2ICogZ3ZbOiwgMToyXSArIGdwWzosIDE6Ml0gLSBOVSAqICh2eHggKyB2eXkpDQogICAgcmV0dXJuIG5zeCwgbnN5LCBndVs6LCAwOjFdICsgZ3ZbOiwgMToyXQ0KDQoNCmRlZiBsb3NzX2ZuKG1vZGVsLCBQLCB2YXJpYW50KToNCiAgICBuc3gsIG5zeSwgY29udCA9IHJlc2lkdWFscyhtb2RlbCwgUFsiY29sIl0pDQogICAgcGh5cyA9IG5zeC5zcXVhcmUoKS5tZWFuKCkgKyBuc3kuc3F1YXJlKCkubWVhbigpICsgY29udC5zcXVhcmUoKS5tZWFuKCkNCiAgICBzZXRzID0gWyJpbmxldCIsICJ0b3AiLCAiYm90dG9tIiwgImN5bCJdDQogICAgYnh5ID0gdG9yY2guY2F0KFtQW2tdIGZvciBrIGluIHNldHNdKQ0KICAgIGJvID0gbW9kZWwoYnh5KQ0KICAgIHR1ID0gdG9yY2guemVyb3MobGVuKGJ4eSksIDEsIGR0eXBlPWJ4eS5kdHlwZSwgZGV2aWNlPWJ4eS5kZXZpY2UpDQogICAgbl9pbiA9IGxlbihQWyJpbmxldCJdKQ0KICAgIHkgPSBQWyJpbmxldCJdWzosIDE6Ml0NCiAgICB0dVs6bl9pbl0gPSA0LjAgKiBVTUFYICogeSAqIChIIC0geSkgLyBIICoqIDINCiAgICBiYyA9IChib1s6LCAwOjFdIC0gdHUpLnNxdWFyZSgpLm1lYW4oKSArIGJvWzosIDE6Ml0uc3F1YXJlKCkubWVhbigpDQogICAgaWYgdmFyaWFudCA9PSAibGl0ZXJhbCI6DQogICAgICAgIGJjID0gYmMgKyBib1s6LCAyOjNdLnNxdWFyZSgpLm1lYW4oKSAgIyBwIHRhcmdldCAwIGF0IGV2ZXJ5IGJvdW5kYXJ5IHBvaW50LCBhcyBpbiB0aGUgdGhlc2lzIGZpZ3VyZQ0KICAgIGVsc2U6DQogICAgICAgIHhvID0gUFsib3V0bGV0Il0uZGV0YWNoKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkNCiAgICAgICAgb28gPSBtb2RlbCh4bykNCiAgICAgICAgZ3UgPSBncmFkKG9vWzosIDA6MV0sIHhvKTsgZ3YgPSBncmFkKG9vWzosIDE6Ml0sIHhvKQ0KICAgICAgICBiYyA9IGJjICsgKE5VICogZ3VbOiwgMDoxXSAtIG9vWzosIDI6M10pLnNxdWFyZSgpLm1lYW4oKSArIChOVSAqIGd2WzosIDA6MV0pLnNxdWFyZSgpLm1lYW4oKQ0KICAgIHJldHVybiBwaHlzICsgYmMsIHsicGh5cyI6IGZsb2F0KHBoeXMuZGV0YWNoKCkpLCAiYmMiOiBmbG9hdChiYy5kZXRhY2goKSl9DQoNCg0KZGVmIHFvaShtb2RlbCwgZGV2KToNCiAgICBuID0gMjA0OA0KICAgIHRoID0gMiAqIG1hdGgucGkgKiB0b3JjaC5hcmFuZ2UobiwgZHR5cGU9dG9yY2guZmxvYXQ2NCwgZGV2aWNlPWRldikgLyBuDQogICAgeHkgPSB0b3JjaC5zdGFjaygoQ1ggKyBSICogdG9yY2guY29zKHRoKSwgQ1kgKyBSICogdG9yY2guc2luKHRoKSksIDEpLnJlcXVpcmVzX2dyYWRfKFRydWUpDQogICAgbyA9IG1vZGVsKHh5KQ0KICAgIGd1LCBndiA9IGdyYWQob1s6LCAwOjFdLCB4eSksIGdyYWQob1s6LCAxOjJdLCB4eSkNCiAgICBueCwgbnkgPSB0b3JjaC5jb3ModGgpLCB0b3JjaC5zaW4odGgpDQogICAgZHMgPSAyICogbWF0aC5waSAqIFIgLyBuDQogICAgcCA9IG9bOiwgMl0NCiAgICBmZCA9IHRvcmNoLnN1bShOVSAqIChndVs6LCAwXSAqIG54ICsgZ3VbOiwgMV0gKiBueSkgLSBwICogbngpICogZHMNCiAgICBmbCA9IHRvcmNoLnN1bShOVSAqIChndls6LCAwXSAqIG54ICsgZ3ZbOiwgMV0gKiBueSkgLSBwICogbnkpICogZHMNCiAgICBzYyA9IDIuMCAvIChVTUVBTiAqKiAyICogRElBTSkNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgcGYgPSBtb2RlbCh0b3JjaC50ZW5zb3IoW1swLjE1LCAwLjJdXSwgZHR5cGU9dG9yY2guZmxvYXQ2NCwgZGV2aWNlPWRldikpWzAsIDJdDQogICAgICAgIHByID0gbW9kZWwodG9yY2gudGVuc29yKFtbMC4yNSwgMC4yXV0sIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYpKVswLCAyXQ0KICAgIHJldHVybiB7IkNkIjogZmxvYXQoc2MgKiBmZCksICJDbCI6IGZsb2F0KHNjICogZmwpLCAiZHAiOiBmbG9hdChwZiAtIHByKX0NCg0KDQpkZWYgc2NvcmUobW9kZWwsIGZlbSwgZGV2KToNCiAgICBtID0gbW9kZWwuZG91YmxlKCkNCiAgICB4eSA9IHRvcmNoLmFzX3RlbnNvcihmZW1bInh5Il0sIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYpDQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6DQogICAgICAgIG8gPSBtKHh5KS5jcHUoKS5udW1weSgpDQogICAgICAgIGFuY2hvciA9IG0odG9yY2gudGVuc29yKFtbTCwgMC4yXV0sIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYpKVswLCAyXS5pdGVtKCkNCiAgICB1LCB2LCBwID0gb1s6LCAwXSwgb1s6LCAxXSwgb1s6LCAyXSAtIGFuY2hvcg0KICAgIHJ1ID0gbGFtYmRhIGEsIGI6IGZsb2F0KG5wLmxpbmFsZy5ub3JtKGEgLSBiKSAvIG5wLmxpbmFsZy5ub3JtKGIpKQ0KICAgIHEgPSBxb2kobSwgZGV2KQ0KICAgIHJldHVybiB7InVfcmVsTDIiOiBydSh1LCBmZW1bInZlbG9jaXR5Il1bOiwgMF0pLCAidl9yZWxMMiI6IHJ1KHYsIGZlbVsidmVsb2NpdHkiXVs6LCAxXSksDQogICAgICAgICAgICAicF9yZWxMMl9nYXVnZSI6IHJ1KHAsIGZlbVsicHJlc3N1cmVfZ2F1Z2VfZml4ZWQiXSksICoqcSwNCiAgICAgICAgICAgICJDZF9yZWxlcnIiOiBhYnMocVsiQ2QiXSAtIEZFQVRbIkNkIl0pIC8gRkVBVFsiQ2QiXSwgIkNsX2Fic2VyciI6IGFicyhxWyJDbCJdIC0gRkVBVFsiQ2wiXSksDQogICAgICAgICAgICAiZHBfcmVsZXJyIjogYWJzKHFbImRwIl0gLSBGRUFUWyJkcCJdKSAvIEZFQVRbImRwIl19DQoNCg0KZGVmIHdvcmtlcihhKToNCiAgICBkZXYgPSB0b3JjaC5kZXZpY2UoYS5kZXZpY2UpDQogICAgb3V0ID0gUGF0aChhLm91dCk7IG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPUZhbHNlKQ0KICAgIHogPSBucC5sb2FkKGEuZmVtKQ0KICAgIGZlbSA9IHtrOiB6W2tdIGZvciBrIGluICgieHkiLCAidmVsb2NpdHkiLCAicHJlc3N1cmVfZ2F1Z2VfZml4ZWQiKX0NCiAgICBkdCA9IHRvcmNoLmZsb2F0MzINCiAgICBQLCBjb3VudHMgPSBtYWtlX3Bvb2xzKGEuc2VlZCwgYS52YXJpYW50LCBkdCwgZGV2KQ0KICAgIHRvcmNoLm1hbnVhbF9zZWVkKGEuc2VlZCkNCiAgICBpZiBhLmFybSA9PSAibWxwIjoNCiAgICAgICAgbW9kZWwgPSBNTFAoWzIsIDUwLCA1MCwgNTAsIDUwLCAzXSkNCiAgICBlbHNlOg0KICAgICAgICBtb2RlbCA9IFJCRktBTigyLCBhLndpZHRoLCAzLCAxMSwgWzAuMCwgMC4wXSwgW0wsIEhdKQ0KICAgIG1vZGVsID0gbW9kZWwudG8oZGV2aWNlPWRldiwgZHR5cGU9ZHQpDQogICAgcGFyYW1zID0gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkpDQogICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbShtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPTFlLTMpDQogICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpOyBoaXN0ID0gW10NCiAgICBmb3IgZXAgaW4gcmFuZ2UoMSwgYS5hZGFtX3N0ZXBzICsgMSk6DQogICAgICAgIG9wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkNCiAgICAgICAgbCwgcGFydHMgPSBsb3NzX2ZuKG1vZGVsLCBQLCBhLnZhcmlhbnQpOyBsLmJhY2t3YXJkKCk7IG9wdC5zdGVwKCkNCiAgICAgICAgaWYgZXAgJSAxMDAgPT0gMCBvciBlcCA9PSBhLmFkYW1fc3RlcHM6DQogICAgICAgICAgICBoaXN0LmFwcGVuZCh7InN0YWdlIjogImFkYW0iLCAic3RlcCI6IGVwLCAibG9zcyI6IGZsb2F0KGwuZGV0YWNoKCkpLCAqKnBhcnRzfSkNCiAgICBpZiBkZXYudHlwZSA9PSAiY3VkYSI6IHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQ0KICAgIHRfYWRhbSA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MA0KICAgIHNkID0ge2s6IHYuZGV0YWNoKCkuY2xvbmUoKSBmb3IgaywgdiBpbiBtb2RlbC5zdGF0ZV9kaWN0KCkuaXRlbXMoKX0NCiAgICB0b3JjaC5zYXZlKHsic3RhdGVfZGljdCI6IHNkLCAic3RhZ2UiOiAiYWRhbV9maW5hbCJ9LCBvdXQgLyAiY2hlY2twb2ludC1hZGFtLWZpbmFsLnB0IikNCiAgICByZXMgPSB7InN0YXR1cyI6IFNUQVRVUywgInZhcmlhbnQiOiBhLnZhcmlhbnQsICJhcm0iOiBhLmFybSwgInNlZWQiOiBhLnNlZWQsICJ3aWR0aCI6IGEud2lkdGggaWYgYS5hcm0gPT0gInJiZiIgZWxzZSBOb25lLA0KICAgICAgICAgICAicGFyYW1zIjogcGFyYW1zLCAiYmNfY291bnRzIjogY291bnRzLCAibl9jb2xsb2NhdGlvbiI6IE5fQ09MLCAiYWRhbV9zdGVwcyI6IGEuYWRhbV9zdGVwcywgImFkYW1fc2Vjb25kcyI6IHRfYWRhbSwNCiAgICAgICAgICAgImRldmljZSI6IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKGRldikgaWYgZGV2LnR5cGUgPT0gImN1ZGEiIGVsc2UgImNwdSIsICJkdHlwZSI6ICJmbG9hdDMyIiwNCiAgICAgICAgICAgImZlbV9yZWZlcmVuY2Vfc2hhMjU2IjogaGFzaGxpYi5zaGEyNTYoUGF0aChhLmZlbSkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwNCiAgICAgICAgICAgInNjb3JlX2FkYW1fZmluYWwiOiBzY29yZShtb2RlbCwgZmVtLCBkZXYpLCAic2NpZW50aWZpY19hY2NlcHRlZCI6IEZhbHNlLCAicmVmZXJlbmNlX2FjY2VwdGVkIjogRmFsc2V9DQogICAgbW9kZWwuZmxvYXQoKQ0KICAgIHByaW50KGpzb24uZHVtcHMoeyJzZWVkIjogYS5zZWVkLCAiYXJtIjogYS5hcm0sICJ2YXJpYW50IjogYS52YXJpYW50LCAiYWRhbSI6IHJlc1sic2NvcmVfYWRhbV9maW5hbCJdfSksIGZsdXNoPVRydWUpDQogICAgbW9kZWwudG8oZHQpDQogICAgbGIgPSB0b3JjaC5vcHRpbS5MQkZHUyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPTEuMCwgbWF4X2l0ZXI9MjUsIG1heF9ldmFsPTEyNSwgdG9sZXJhbmNlX2dyYWQ9MWUtOCwgdG9sZXJhbmNlX2NoYW5nZT0wLjAsDQogICAgICAgICAgICAgICAgICAgICAgICAgICBoaXN0b3J5X3NpemU9MTAwLCBsaW5lX3NlYXJjaF9mbj0ic3Ryb25nX3dvbGZlIikNCiAgICB0MSA9IHRpbWUucGVyZl9jb3VudGVyKCk7IHN0YXR1cyA9ICJsYmZnc190aW1lX2NhcCI7IGl0ZXJzID0gMDsgcGxhdGVhdSA9IDANCiAgICBkZWYgY2xvc3VyZSgpOg0KICAgICAgICBsYi56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkNCiAgICAgICAgbCwgXyA9IGxvc3NfZm4obW9kZWwsIFAsIGEudmFyaWFudCk7IGwuYmFja3dhcmQoKTsgcmV0dXJuIGwNCiAgICBsYXN0ID0gZmxvYXQobG9zc19mbihtb2RlbCwgUCwgYS52YXJpYW50KVswXS5kZXRhY2goKSkNCiAgICBhY2NlcHRlZCA9IHtrOiB2LmRldGFjaCgpLmNsb25lKCkgZm9yIGssIHYgaW4gbW9kZWwuc3RhdGVfZGljdCgpLml0ZW1zKCl9DQogICAgd2hpbGUgdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQxIDwgYS5sYmZnc19zZWNvbmRzIGFuZCBpdGVycyA8IGEubGJmZ3NfbWF4X2l0ZXI6DQogICAgICAgIGJlZm9yZSA9IGxiLnN0YXRlW2xiLl9wYXJhbXNbMF1dLmdldCgibl9pdGVyIiwgMCkNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgbGIuc3RlcChjbG9zdXJlKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAjIG5vcWE6IEJMRTAwMQ0KICAgICAgICAgICAgc3RhdHVzID0gZiJsYmZnc19leGNlcHRpb246e3R5cGUoZSkuX19uYW1lX199IjsgYnJlYWsNCiAgICAgICAgY3VyID0gZmxvYXQobG9zc19mbihtb2RlbCwgUCwgYS52YXJpYW50KVswXS5kZXRhY2goKSkNCiAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoY3VyKSBvciBjdXIgPiBsYXN0ICogMS41Og0KICAgICAgICAgICAgc3RhdHVzID0gImxiZmdzX3JlamVjdGVkX25vbmZpbml0ZV9vcl93b3JzZSI7IGJyZWFrDQogICAgICAgIGQgPSBsYi5zdGF0ZVtsYi5fcGFyYW1zWzBdXS5nZXQoIm5faXRlciIsIDApIC0gYmVmb3JlDQogICAgICAgIGltcHJvdmVkID0gKGxhc3QgLSBjdXIpIC8gbWF4KGFicyhsYXN0KSwgMWUtMzApDQogICAgICAgIGFjY2VwdGVkID0ge2s6IHYuZGV0YWNoKCkuY2xvbmUoKSBmb3IgaywgdiBpbiBtb2RlbC5zdGF0ZV9kaWN0KCkuaXRlbXMoKX0NCiAgICAgICAgbGFzdCA9IGN1cjsgaXRlcnMgKz0gZA0KICAgICAgICBoaXN0LmFwcGVuZCh7InN0YWdlIjogImxiZmdzIiwgInN0ZXAiOiBpdGVycywgImxvc3MiOiBjdXJ9KQ0KICAgICAgICBpZiBkID09IDA6DQogICAgICAgICAgICBzdGF0dXMgPSAibGJmZ3NfemVyb19wcm9ncmVzcyI7IGJyZWFrDQogICAgICAgIHBsYXRlYXUgPSBwbGF0ZWF1ICsgMSBpZiBpbXByb3ZlZCA8IDFlLTEwIGVsc2UgMA0KICAgICAgICBpZiBwbGF0ZWF1ID49IDU6DQogICAgICAgICAgICBzdGF0dXMgPSAibGJmZ3NfcGxhdGVhdSI7IGJyZWFrDQogICAgZWxzZToNCiAgICAgICAgc3RhdHVzID0gImxiZmdzX2l0ZXJfY2FwIiBpZiBpdGVycyA+PSBhLmxiZmdzX21heF9pdGVyIGVsc2UgImxiZmdzX3RpbWVfY2FwIg0KICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChhY2NlcHRlZCkNCiAgICBpZiBkZXYudHlwZSA9PSAiY3VkYSI6IHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQ0KICAgIHJlcy51cGRhdGUoeyJsYmZnc19zdGF0dXMiOiBzdGF0dXMsICJsYmZnc19zZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQxLCAibGJmZ3NfaXRlcmF0aW9ucyI6IGl0ZXJzLCAibGJmZ3Nfc2Vjb25kc19jYXAiOiBhLmxiZmdzX3NlY29uZHN9KQ0KICAgIHRvcmNoLnNhdmUoeyJzdGF0ZV9kaWN0IjogbW9kZWwuc3RhdGVfZGljdCgpLCAic3RhZ2UiOiAibGJmZ3NfZmluYWwifSwgb3V0IC8gImNoZWNrcG9pbnQtbGJmZ3MtZmluYWwucHQiKQ0KICAgIHJlc1sic2NvcmVfbGJmZ3NfZmluYWwiXSA9IHNjb3JlKG1vZGVsLCBmZW0sIGRldikNCiAgICAob3V0IC8gImhpc3RvcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhoaXN0KSkNCiAgICAob3V0IC8gInJ1bi5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcywgaW5kZW50PTIpKQ0KICAgIHByaW50KGpzb24uZHVtcHMoeyJzZWVkIjogYS5zZWVkLCAiYXJtIjogYS5hcm0sICJ2YXJpYW50IjogYS52YXJpYW50LCAibGJmZ3MiOiByZXNbInNjb3JlX2xiZmdzX2ZpbmFsIl0sICJzdGF0dXMiOiBzdGF0dXN9KSwgZmx1c2g9VHJ1ZSkNCg0KDQpkZWYgY2FtcGFpZ24oYSk6DQogICAgb3V0ID0gUGF0aChhLm91dCk7IG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgbl9ncHUgPSBtYXgoMSwgdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSkNCiAgICBxdWV1ZSA9IFsodiwgYXJtLCBzKSBmb3IgcyBpbiBhLnNlZWRzIGZvciB2IGluIGEudmFyaWFudHMgZm9yIGFybSBpbiBhLmFybXNdDQogICAgcnVubmluZywgbGVkZ2VyLCB0MCA9IHt9LCBbXSwgdGltZS5tb25vdG9uaWMoKQ0KICAgIHdoaWxlIHF1ZXVlIG9yIHJ1bm5pbmc6DQogICAgICAgIGZvciBnIGluIGxpc3QocnVubmluZyk6DQogICAgICAgICAgICBwciwgam9iLCBzdCA9IHJ1bm5pbmdbZ10NCiAgICAgICAgICAgIGlmIHByLnBvbGwoKSBpcyBub3QgTm9uZToNCiAgICAgICAgICAgICAgICBsZWRnZXIuYXBwZW5kKHsiam9iIjogam9iLCAicmMiOiBwci5yZXR1cm5jb2RlLCAid2FsbCI6IHRpbWUubW9ub3RvbmljKCkgLSBzdH0pOyBkZWwgcnVubmluZ1tnXQ0KICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gdDAgPiBhLndhbGxfY2FwOiBxdWV1ZSA9IFtdDQogICAgICAgIGZvciBnIGluIHJhbmdlKG1pbihuX2dwdSwgMikpOg0KICAgICAgICAgICAgaWYgZyBub3QgaW4gcnVubmluZyBhbmQgcXVldWU6DQogICAgICAgICAgICAgICAgdiwgYXJtLCBzID0gcXVldWUucG9wKDApDQogICAgICAgICAgICAgICAgdGFnID0gZiJ7dn0te2FybX0tc2VlZC17c30iDQogICAgICAgICAgICAgICAgY21kID0gW3N5cy5leGVjdXRhYmxlLCBfX2ZpbGVfXywgIndvcmtlciIsICItLXZhcmlhbnQiLCB2LCAiLS1hcm0iLCBhcm0sICItLXNlZWQiLCBzdHIocyksICItLXdpZHRoIiwgc3RyKGEud2lkdGgpLA0KICAgICAgICAgICAgICAgICAgICAgICAiLS1kZXZpY2UiLCBmImN1ZGE6e2d9IiwgIi0tZmVtIiwgYS5mZW0sICItLW91dCIsIHN0cihvdXQgLyB0YWcpLCAiLS1hZGFtLXN0ZXBzIiwgc3RyKGEuYWRhbV9zdGVwcyksDQogICAgICAgICAgICAgICAgICAgICAgICItLWxiZmdzLXNlY29uZHMiLCBzdHIoYS5sYmZnc19zZWNvbmRzKSwgIi0tbGJmZ3MtbWF4LWl0ZXIiLCBzdHIoYS5sYmZnc19tYXhfaXRlcildDQogICAgICAgICAgICAgICAgcnVubmluZ1tnXSA9IChzdWJwcm9jZXNzLlBvcGVuKGNtZCwgc3Rkb3V0PW9wZW4ob3V0IC8gZiJ7dGFnfS5sb2ciLCAidyIpLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpLCB0YWcsIHRpbWUubW9ub3RvbmljKCkpDQogICAgICAgIChvdXQgLyAibGVkZ2VyLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyJzdGF0dXMiOiBTVEFUVVMsICJydW5zIjogbGVkZ2VyLCAicnVubmluZyI6IFtqIGZvciBfLCBqLCBfIGluIHJ1bm5pbmcudmFsdWVzKCldfSkpDQogICAgICAgIHRpbWUuc2xlZXAoMikNCiAgICAob3V0IC8gImxlZGdlci5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHsic3RhdHVzIjogU1RBVFVTLCAicnVucyI6IGxlZGdlciwgImNvbXBsZXRlIjogVHJ1ZSwgIndhbGwiOiB0aW1lLm1vbm90b25pYygpIC0gdDB9KSkNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgibW9kZSIsIGNob2ljZXM9KCJ3b3JrZXIiLCAiY2FtcGFpZ24iKSkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdmFyaWFudCIsIGNob2ljZXM9KCJsaXRlcmFsIiwgImNvcnJlY3RlZCIpLCBkZWZhdWx0PSJjb3JyZWN0ZWQiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS12YXJpYW50cyIsIG5hcmdzPSIrIiwgZGVmYXVsdD1bImNvcnJlY3RlZCJdKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1hcm0iLCBjaG9pY2VzPSgibWxwIiwgInJiZiIpLCBkZWZhdWx0PSJtbHAiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1hcm1zIiwgbmFyZ3M9IisiLCBkZWZhdWx0PVsibWxwIiwgInJiZiJdKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkcyIsIG5hcmdzPSIrIiwgdHlwZT1pbnQsIGRlZmF1bHQ9WzgwMSwgODAyLCA4MDMsIDgwNCwgODA1XSkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTgwMSkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td2lkdGgiLCB0eXBlPWludCwgZGVmYXVsdD0xNikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD0iY3VkYTowIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZmVtIiwgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYWRhbS1zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTIwMDAwKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1sYmZncy1zZWNvbmRzIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD02MDAuMCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbGJmZ3MtbWF4LWl0ZXIiLCB0eXBlPWludCwgZGVmYXVsdD0xNTAwMCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td2FsbC1jYXAiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTFlOSkNCiAgICBhID0gYXAucGFyc2VfYXJncygpDQogICAgeyJ3b3JrZXIiOiB3b3JrZXIsICJjYW1wYWlnbiI6IGNhbXBhaWdufVthLm1vZGVdKGEpDQo=', 'dfg_L3_fields.npz': '...'}
for k, v in BLOB.items():
    (p / k).write_bytes(base64.b64decode(v))
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(json.dumps({"gpus": names, "torch": torch.__version__, "hashes": {'pikan_campaign.py': '3a2c836ea3615b4a933a5c35550a6f80cc269ffd1e5cac2623fa82b86b84b619', 'dfg_long.py': '792bcbf53f2453eef7598fd4a20f0c51bd0c7fe9ca1ea1b294b5f5c8d8d59b84', 'dfg_L3_fields.npz': '936fc178e6bc63b1c72388b76c269e2eb56ff78a93c14903df512a7d7202bb7a'}}), flush=True)
assert names and all("T4" in x for x in names), names
cmd = [sys.executable, str(p / "dfg_long.py"), "campaign", "--fem", str(p / "dfg_L3_fields.npz"),
       "--out", str(w / "results"), "--wall-cap", str(START_CUTOFF), "--width", "16",
       "--seeds", "801", "802", "803", "804", "805"]
proc = subprocess.Popen(cmd, stdout=open(w / "runner.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
timed_out = False
while proc.poll() is None:
    if time.monotonic() - T0 > CAP:
        timed_out = True; os.killpg(proc.pid, signal.SIGTERM); time.sleep(10)
        try: os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        break
    time.sleep(3)
summ = {"status": "timeout_stopped" if timed_out else "complete", "rc": proc.wait(), "wall": time.monotonic() - T0,
        "cap": CAP, "gpus": names, "scientific_accepted": False, "reference_accepted": False}
(w / "session_summary.json").write_text(json.dumps(summ, indent=1)); print(json.dumps(summ))
